# Modifica su MultiPICo


Aggiunta degli score per semi-aggregazione su prospettiva (gender e generation)

In [2]:
import json
import random
from collections import Counter
from typing import Any, Dict, Optional, Tuple

INPUT_PATH = "MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json"
OUTPUT_PATH = "MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered_with_semiaggregates.json"

TARGET_FIELD = "irony_label"
GROUP_FIELDS = ("gender", "generation")
STORE_STATS = True

SEED = 42          # metti None se vuoi random puro
if SEED is not None:
    random.seed(SEED)


def majority_vote_with_random_tie(values) -> Tuple[Optional[int], Dict[int, int]]:
    """
    - len == 1 -> ritorna quella label
    - majority -> ritorna majority
    - pareggio -> random choice tra le label in pareggio
    """
    if not values:
        return None, {}

    c = Counter(values)

    if len(values) == 1:
        return values[0], dict(c)

    most_common = c.most_common()
    max_count = most_common[0][1]

    tied_labels = [label for label, cnt in most_common if cnt == max_count]

    if len(tied_labels) == 1:
        return tied_labels[0], dict(c)

    # pareggio → sorteggio
    return random.choice(tied_labels), dict(c)


def slug(x: Any) -> str:
    return str(x).strip().lower().replace(" ", "_")


def add_semiaggregates(data: Dict[str, Any]) -> Dict[str, Any]:
    for entry in data.get("entries", []) or []:
        ann = entry.get("annotations", {}) or {}
        observers = ann.get("observers_annotations", []) or []

        for gf in GROUP_FIELDS:
            buckets = {}
            for ob in observers:
                gval = ob.get(gf)
                if gval is None:
                    continue
                lbl = ob.get(TARGET_FIELD)
                if lbl is None:
                    continue
                buckets.setdefault(gval, []).append(lbl)

            for gval, labels in buckets.items():
                maj, counts = majority_vote_with_random_tie(labels)
                key = f"aggregation_on_{slug(gval)}"
                ann[key] = maj

                if STORE_STATS:
                    ann[f"{key}_n"] = len(labels)
                    for k, v in counts.items():
                        ann[f"{key}_votes_{k}"] = v

        entry["annotations"] = ann

    return data


def main():
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    data = add_semiaggregates(data)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"OK! Salvato JSON con tie-break random:\n{OUTPUT_PATH}")


if __name__ == "__main__":
    main()

OK! Salvato JSON con tie-break random:
MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered_with_semiaggregates.json


# Modifica su CSC

Aggiunta della label generation sulla base dell'age prendendo come anno di riferimento il 2024

In [3]:
import json
from typing import Any, Dict, Optional

INPUT_PATH = "CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered.json"
OUTPUT_PATH = "CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered_with_generation.json"

REFERENCE_YEAR = 2024  # come richiesto


def generation_from_age(age: Any, reference_year: int = REFERENCE_YEAR) -> Optional[str]:
    if age is None:
        return None
    try:
        age_int = int(age)
    except (TypeError, ValueError):
        return None

    birth_year = reference_year - age_int

    if 1946 <= birth_year <= 1964:
        return "Boomer"
    if 1965 <= birth_year <= 1980:
        return "GenX"
    if 1981 <= birth_year <= 1996:
        return "GenY"
    if 1997 <= birth_year <= 2012:
        return "GenZ"
    return None  # fuori range (es. <1946 o >2012)


def fill_generation_in_annotator(ann: Dict[str, Any]) -> None:
    # aggiorna solo se generation è null/mancante
    if ann.get("generation") is not None:
        return
    gen = generation_from_age(ann.get("age"))
    if gen is not None:
        ann["generation"] = gen


def main():
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    for entry in data.get("entries", []) or []:
        annotations = entry.get("annotations", {}) or {}

        # speaker_annotation
        sp = annotations.get("speaker_annotation")
        if isinstance(sp, dict):
            fill_generation_in_annotator(sp)
            annotations["speaker_annotation"] = sp

        # observers_annotations
        obs = annotations.get("observers_annotations")
        if isinstance(obs, list):
            for ob in obs:
                if isinstance(ob, dict):
                    fill_generation_in_annotator(ob)
            annotations["observers_annotations"] = obs

        entry["annotations"] = annotations

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"OK! Salvato JSON: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

OK! Salvato JSON: CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered_with_generation.json


In [ ]:
Aggiunta degli score per semi-aggregazione su prospettiva (gender e generation)\

In [6]:
import json
import random
from collections import Counter
from typing import Any, Dict, Optional, Tuple

INPUT_PATH = "CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered_with_generation.json"
OUTPUT_PATH = "CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered_with_semiaggregates.json"

TARGET_FIELD = "sarcasm_label"
GROUP_FIELDS = ("gender", "generation")
STORE_STATS = True

SEED = 42          # metti None se vuoi random puro
if SEED is not None:
    random.seed(SEED)


def majority_vote_with_random_tie(values) -> Tuple[Optional[int], Dict[int, int]]:
    """
    - len == 1 -> ritorna quella label
    - majority -> ritorna majority
    - pareggio -> random choice tra le label in pareggio
    """
    if not values:
        return None, {}

    c = Counter(values)

    if len(values) == 1:
        return values[0], dict(c)

    most_common = c.most_common()
    max_count = most_common[0][1]

    tied_labels = [label for label, cnt in most_common if cnt == max_count]

    if len(tied_labels) == 1:
        return tied_labels[0], dict(c)

    # pareggio → sorteggio
    return random.choice(tied_labels), dict(c)


def slug(x: Any) -> str:
    return str(x).strip().lower().replace(" ", "_")


def add_semiaggregates(data: Dict[str, Any]) -> Dict[str, Any]:
  for entry in data.get("entries", []) or []:
      ann = entry.get("annotations", {}) or {}
      observers = ann.get("observers_annotations", []) or []

      # ✅ TEMP: conta anche lo speaker nel calcolo, senza modificarlo nel JSON
      speaker = ann.get("speaker_annotation")
      observers_for_agg = list(observers)
      if isinstance(speaker, dict):
          observers_for_agg.append(speaker)

      for gf in GROUP_FIELDS:
          buckets = {}
          for ob in observers_for_agg:   # <-- qui usi la lista temporanea
              gval = ob.get(gf)
              if gval is None:
                  continue
              lbl = ob.get(TARGET_FIELD)
              if lbl is None:
                  continue
              buckets.setdefault(gval, []).append(lbl)

          for gval, labels in buckets.items():
              maj, counts = majority_vote_with_random_tie(labels)
              key = f"aggregation_on_{slug(gval)}"
              ann[key] = maj

              if STORE_STATS:
                  ann[f"{key}_n"] = len(labels)
                  for k, v in counts.items():
                      ann[f"{key}_votes_{k}"] = v

      entry["annotations"] = ann

  return data

def main():
    with open(INPUT_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    data = add_semiaggregates(data)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"OK! Salvato JSON con tie-break random:\n{OUTPUT_PATH}")


if __name__ == "__main__":
    main()

OK! Salvato JSON con tie-break random:
CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_filtered_with_semiaggregates.json
